# AeroTrack — VisDrone MOT Image Sequences to MP4

This notebook reconstructs the image folders in `VisDrone2019-MOT-val` into one MP4 per sequence.

Your current validation folder contains seven sequences:
- `uav0000086_00000_v`
- `uav0000117_02622_v`
- `uav0000137_00458_v`
- `uav0000182_00000_v`
- `uav0000268_05773_v`
- `uav0000305_00000_v`
- `uav0000339_00001_v`

Each sequence becomes a separate MP4. The original image folders and annotation files are left unchanged.

**Important:** the images do not contain their original timestamps. `OUTPUT_FPS` below sets the playback rate of the reconstructed video. The default 30 FPS is a working value, not a claim that every original clip was recorded at 30 FPS. Detection still works on the ordered frames, but time/duration measurements depend on the chosen FPS.

The generated videos are clean (no ground-truth boxes drawn on them), so they can be used as input to AeroTrack Part 1.

## Section 1 — Configure folders and video settings

The path search checks the nested folder structure shown in your Windows screenshot first.

If the automatic search does not find the dataset, set `VISDRONE_ROOT` manually to the folder that directly contains both `sequences` and `annotations`.

In [1]:
from pathlib import Path
import re
import cv2
import pandas as pd
import numpy as np
from IPython.display import display, Video

# Search common locations, including the nested folder from your screenshot.
download_dir = Path.home() / "Downloads"

possible_roots = [
    download_dir / "VisDrone2019-MOT-val" / "VisDrone2019-MOT-val",
    download_dir / "VisDrone2019-MOT-val",
    Path.cwd() / "VisDrone2019-MOT-val",
]

VISDRONE_ROOT = next(
    (
        path for path in possible_roots
        if (path / "sequences").is_dir()
        and (path / "annotations").is_dir()
    ),
    None
)

# If automatic detection failed, uncomment and edit this line:
# VISDRONE_ROOT = Path(r"C:\Users\anyan\Downloads\VisDrone2019-MOT-val\VisDrone2019-MOT-val")

if VISDRONE_ROOT is None:
    raise FileNotFoundError(
        "Could not automatically find the VisDrone MOT folder. "
        "Set VISDRONE_ROOT manually to the folder containing 'sequences' "
        "and 'annotations'."
    )

SEQUENCES_DIR = VISDRONE_ROOT / "sequences"
ANNOTATIONS_DIR = VISDRONE_ROOT / "annotations"

# Save reconstructed videos inside the AeroTrack project.
AEROTRACK_ROOT = Path.cwd()
OUTPUT_DIR = AEROTRACK_ROOT / "data" / "visdrone_reconstructed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This is the reconstructed video's playback rate, not a recovered source timestamp.
OUTPUT_FPS = 30.0

# Set to None to convert every sequence, or provide a list of sequence folder names.
SEQUENCES_TO_CONVERT = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

print("VisDrone root:", VISDRONE_ROOT)
print("Sequences folder exists:", SEQUENCES_DIR.exists())
print("Annotations folder exists:", ANNOTATIONS_DIR.exists())
print("AeroTrack output folder:", OUTPUT_DIR)
print("Output FPS:", OUTPUT_FPS)

VisDrone root: C:\Users\anyan\Downloads\VisDrone2019-MOT-val\VisDrone2019-MOT-val
Sequences folder exists: True
Annotations folder exists: True
AeroTrack output folder: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed
Output FPS: 30.0


## Section 2 — Inspect all sequences and matching annotations

This cell counts images in each sequence and checks whether a matching annotation text file exists.

It also reads the annotation frame numbers. Those numbers will be mapped to the sequential frame positions of each reconstructed MP4.

In [2]:
def numeric_frame_number(image_path):
    """Return the final numeric part of an image filename, if present."""
    match = re.search(r"(\d+)$", image_path.stem)
    return int(match.group(1)) if match else None


def frame_sort_key(image_path):
    """Sort frame files numerically rather than alphabetically."""
    frame_number = numeric_frame_number(image_path)

    if frame_number is not None:
        return (0, frame_number, image_path.name.lower())

    return (1, image_path.name.lower())


sequence_folders = sorted(
    path for path in SEQUENCES_DIR.iterdir()
    if path.is_dir()
)

if SEQUENCES_TO_CONVERT is not None:
    selected = set(SEQUENCES_TO_CONVERT)
    sequence_folders = [
        path for path in sequence_folders
        if path.name in selected
    ]

if not sequence_folders:
    raise FileNotFoundError("No sequence folders were found to convert.")

inspection_rows = []

for sequence_folder in sequence_folders:
    image_files = sorted(
        [
            path for path in sequence_folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ],
        key=frame_sort_key
    )

    annotation_path = ANNOTATIONS_DIR / f"{sequence_folder.name}.txt"

    first_number = (
        numeric_frame_number(image_files[0])
        if image_files else None
    )
    last_number = (
        numeric_frame_number(image_files[-1])
        if image_files else None
    )

    annotation_rows = 0
    annotation_frame_matches = 0

    if annotation_path.exists() and annotation_path.stat().st_size > 0:
        annotation_df = pd.read_csv(
            annotation_path,
            header=None,
            on_bad_lines="skip"
        )
        annotation_rows = len(annotation_df)

        if not annotation_df.empty and first_number is not None:
            image_frame_numbers = {
                numeric_frame_number(path)
                for path in image_files
                if numeric_frame_number(path) is not None
            }

            annotation_frame_numbers = set(
                pd.to_numeric(
                    annotation_df.iloc[:, 0],
                    errors="coerce"
                ).dropna().astype(int)
            )

            annotation_frame_matches = len(
                image_frame_numbers.intersection(annotation_frame_numbers)
            )

    inspection_rows.append({
        "sequence": sequence_folder.name,
        "images": len(image_files),
        "first_image_number": first_number,
        "last_image_number": last_number,
        "annotation_file_exists": annotation_path.exists(),
        "annotation_rows": annotation_rows,
        "annotation_frame_numbers_matching_images": annotation_frame_matches
    })

inspection = pd.DataFrame(inspection_rows)
display(inspection)

print("Sequences found:", len(inspection))
print("Total image frames:", int(inspection["images"].sum()))

,sequence,images,first_image_number,last_image_number,annotation_file_exists,annotation_rows,annotation_frame_numbers_matching_images
0,uav0000086_00000_v,464,1,464,True,23877,464
1,uav0000117_02622_v,349,1,349,True,18463,349
2,uav0000137_00458_v,233,1,233,True,24361,233
3,uav0000182_00000_v,363,1,363,True,18441,363
4,uav0000268_05773_v,978,1,978,True,13578,978
5,uav0000305_00000_v,184,1,184,True,6391,184
6,uav0000339_00001_v,275,1,275,True,13016,275


Sequences found: 7
Total image frames: 2846


## Section 3 — Reconstruct each sequence into an MP4

The converter:
1. Sorts image files by their numeric frame number.
2. Reads one frame at a time (it does not load the entire sequence into memory).
3. Checks that all images have the same dimensions.
4. Resizes an inconsistent frame to the first frame's dimensions if necessary.
5. Writes a clean MP4 using `mp4v`.
6. Records the relationship between each source image number and the new MP4 frame number.

If MP4 encoding is unavailable in your OpenCV installation, the cell falls back to AVI/XVID.

In [3]:
def list_sequence_images(sequence_folder):
    return sorted(
        [
            path for path in sequence_folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ],
        key=frame_sort_key
    )


def create_video_from_sequence(sequence_folder, output_folder, fps):
    image_files = list_sequence_images(sequence_folder)

    if not image_files:
        raise RuntimeError(f"No images found in {sequence_folder}")

    # Read the first valid image to determine video dimensions.
    first_frame = None
    first_image_path = None

    for image_path in image_files:
        first_frame = cv2.imread(str(image_path))
        if first_frame is not None:
            first_image_path = image_path
            break

    if first_frame is None:
        raise RuntimeError(f"Could not read any images in {sequence_folder}")

    height, width = first_frame.shape[:2]
    sequence_name = sequence_folder.name

    output_folder.mkdir(parents=True, exist_ok=True)

    mp4_path = output_folder / f"{sequence_name}.mp4"
    avi_path = output_folder / f"{sequence_name}.avi"

    # Try MP4 first.
    writer = cv2.VideoWriter(
        str(mp4_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        float(fps),
        (width, height)
    )

    actual_output_path = mp4_path

    # Fall back to AVI/XVID if MP4 is not supported by this OpenCV build.
    if not writer.isOpened():
        writer.release()
        if mp4_path.exists():
            mp4_path.unlink()

        writer = cv2.VideoWriter(
            str(avi_path),
            cv2.VideoWriter_fourcc(*"XVID"),
            float(fps),
            (width, height)
        )
        actual_output_path = avi_path

    if not writer.isOpened():
        writer.release()
        raise RuntimeError(
            "OpenCV could not initialize MP4 or AVI encoding. "
            "Check your OpenCV/FFmpeg installation."
        )

    mapping_rows = []
    written_frames = 0
    resized_frames = 0
    unreadable_frames = []

    try:
        for image_path in image_files:
            frame = cv2.imread(str(image_path))

            if frame is None:
                unreadable_frames.append(image_path.name)
                continue

            original_height, original_width = frame.shape[:2]

            if (original_width, original_height) != (width, height):
                frame = cv2.resize(
                    frame,
                    (width, height),
                    interpolation=cv2.INTER_AREA
                )
                resized_frames += 1

            writer.write(frame)
            written_frames += 1

            source_frame_number = numeric_frame_number(image_path)

            mapping_rows.append({
                "video_id": sequence_name,
                "source_image": image_path.name,
                "source_frame_number": source_frame_number,
                # This is the 1-based frame number inside the new MP4.
                "video_frame_number": written_frames,
                "time_seconds_nominal": written_frames / float(fps)
            })

    finally:
        writer.release()

    if written_frames == 0:
        raise RuntimeError(f"No frames were written for {sequence_name}")

    result = {
        "video_id": sequence_name,
        "video_path": str(actual_output_path),
        "frames_written": written_frames,
        "fps": float(fps),
        "width": width,
        "height": height,
        "duration_seconds_nominal": written_frames / float(fps),
        "resized_frames": resized_frames,
        "unreadable_frames": len(unreadable_frames),
        "first_source_frame": numeric_frame_number(image_files[0]),
        "last_source_frame": numeric_frame_number(image_files[-1]),
        "annotation_path": str(
            ANNOTATIONS_DIR / f"{sequence_name}.txt"
        )
    }

    mapping = pd.DataFrame(mapping_rows)

    return result, mapping, unreadable_frames


conversion_results = []
frame_mappings = []
conversion_errors = []

for index, sequence_folder in enumerate(sequence_folders, start=1):
    print(
        f"\n[{index}/{len(sequence_folders)}] "
        f"Converting {sequence_folder.name} ..."
    )

    try:
        result, mapping, unreadable = create_video_from_sequence(
            sequence_folder=sequence_folder,
            output_folder=OUTPUT_DIR,
            fps=OUTPUT_FPS
        )

        conversion_results.append(result)
        frame_mappings.append(mapping)

        print("Saved:", result["video_path"])
        print("Frames written:", result["frames_written"])
        print("Resolution:", f"{result['width']} x {result['height']}")
        print("Nominal duration:", round(result["duration_seconds_nominal"], 2), "seconds")
        print("Resized frames:", result["resized_frames"])
        print("Unreadable frames:", result["unreadable_frames"])

        if unreadable:
            print("First unreadable files:", unreadable[:5])

    except Exception as exc:
        conversion_errors.append({
            "sequence": sequence_folder.name,
            "error": str(exc)
        })
        print("ERROR:", exc)

if not conversion_results:
    raise RuntimeError("No sequences were successfully converted.")

conversion_manifest = pd.DataFrame(conversion_results)
frame_mapping = pd.concat(frame_mappings, ignore_index=True)

MANIFEST_PATH = OUTPUT_DIR / "visdrone_video_manifest.csv"
FRAME_MAPPING_PATH = OUTPUT_DIR / "visdrone_frame_mapping.csv"
ERRORS_PATH = OUTPUT_DIR / "visdrone_conversion_errors.csv"

conversion_manifest.to_csv(MANIFEST_PATH, index=False)
frame_mapping.to_csv(FRAME_MAPPING_PATH, index=False)

pd.DataFrame(
    conversion_errors,
    columns=["sequence", "error"]
).to_csv(ERRORS_PATH, index=False)

print("\nCONVERSION SUMMARY")
display(conversion_manifest)

print("Manifest:", MANIFEST_PATH)
print("Frame mapping:", FRAME_MAPPING_PATH)
print("Errors:", ERRORS_PATH)


[1/7] Converting uav0000086_00000_v ...
Saved: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\uav0000086_00000_v.mp4
Frames written: 464
Resolution: 1344 x 756
Nominal duration: 15.47 seconds
Resized frames: 0
Unreadable frames: 0

[2/7] Converting uav0000117_02622_v ...
Saved: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\uav0000117_02622_v.mp4
Frames written: 349
Resolution: 2720 x 1530
Nominal duration: 11.63 seconds
Resized frames: 0
Unreadable frames: 0

[3/7] Converting uav0000137_00458_v ...
Saved: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\uav0000137_00458_v.mp4
Frames written: 233
Resolution: 2688 x 1512
Nominal duration: 7.77 seconds
Resized frames: 0
Unreadable frames: 0

[4/7] Converting uav0000182_00000_v ...
Saved: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\uav0000182_00000_v.mp4
Frames written: 363
Resolution: 1344 x 756
Nominal duration: 12.1 seconds
Resized frames: 0
Unreadable frames

,video_id,video_path,frames_written,fps,width,height,duration_seconds_nominal,resized_frames,unreadable_frames,first_source_frame,last_source_frame,annotation_path
0,uav0000086_00000_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,464,30.0,1344,756,15.466667,0,0,1,464,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
1,uav0000117_02622_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,349,30.0,2720,1530,11.633333,0,0,1,349,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
2,uav0000137_00458_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,233,30.0,2688,1512,7.766667,0,0,1,233,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
3,uav0000182_00000_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,363,30.0,1344,756,12.100000,0,0,1,363,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
4,uav0000268_05773_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,978,30.0,3840,2160,32.600000,0,0,1,978,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
5,uav0000305_00000_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,184,30.0,1904,1071,6.133333,0,0,1,184,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...
6,uav0000339_00001_v,c:\Users\anyan\source\repos\AeroTrack\data\vis...,275,30.0,1904,1071,9.166667,0,0,1,275,C:\Users\anyan\Downloads\VisDrone2019-MOT-val\...


Manifest: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\visdrone_video_manifest.csv
Frame mapping: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\visdrone_frame_mapping.csv
Errors: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\visdrone_conversion_errors.csv


## Section 4 — Validate that the MP4 files can be opened

This cell reopens every converted video and checks:
- the video can be decoded,
- the output frame count,
- the output FPS,
- the output dimensions.

The reported duration is based on the configured output FPS.

In [4]:
validation_rows = []

for _, row in conversion_manifest.iterrows():
    video_path = Path(row["video_path"])
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        validation_rows.append({
            "video_id": row["video_id"],
            "video_opens": False,
            "reported_frames": 0,
            "reported_fps": np.nan,
            "width": np.nan,
            "height": np.nan,
            "status": "FAILED_TO_OPEN"
        })
        continue

    reported_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    reported_fps = float(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Read the first frame to confirm actual decoding.
    success, first_frame = cap.read()
    cap.release()

    validation_rows.append({
        "video_id": row["video_id"],
        "video_opens": bool(success and first_frame is not None),
        "reported_frames": reported_frames,
        "frames_written": int(row["frames_written"]),
        "reported_fps": reported_fps,
        "width": width,
        "height": height,
        "status": (
            "OK"
            if success and reported_frames == int(row["frames_written"])
            else "CHECK"
        )
    })

validation = pd.DataFrame(validation_rows)
display(validation)

VALIDATION_PATH = OUTPUT_DIR / "visdrone_video_validation.csv"
validation.to_csv(VALIDATION_PATH, index=False)

if (validation["status"] != "OK").any():
    print("Some videos need inspection. Check:", VALIDATION_PATH)
else:
    print("All reconstructed videos passed the basic validation.")

,video_id,video_opens,reported_frames,frames_written,reported_fps,width,height,status
0,uav0000086_00000_v,True,464,464,30.0,1344,756,OK
1,uav0000117_02622_v,True,349,349,30.0,2720,1530,OK
2,uav0000137_00458_v,True,233,233,30.0,2688,1512,OK
3,uav0000182_00000_v,True,363,363,30.0,1344,756,OK
4,uav0000268_05773_v,True,978,978,30.0,3840,2160,OK
5,uav0000305_00000_v,True,184,184,30.0,1904,1070,OK
6,uav0000339_00001_v,True,275,275,30.0,1904,1070,OK


All reconstructed videos passed the basic validation.


## Section 5 — Preview one reconstructed video

Change `PREVIEW_SEQUENCE` to any sequence name in the manifest.

The preview is the clean reconstructed MP4, without ground-truth boxes.

In [5]:
PREVIEW_SEQUENCE = conversion_manifest.iloc[0]["video_id"]

preview_row = conversion_manifest[
    conversion_manifest["video_id"] == PREVIEW_SEQUENCE
].iloc[0]

preview_path = Path(preview_row["video_path"])

print("Previewing:", PREVIEW_SEQUENCE)
print("Video:", preview_path)
print("Frames:", preview_row["frames_written"])
print("FPS:", preview_row["fps"])
print("Resolution:", preview_row["width"], "x", preview_row["height"])

display(Video(
    str(preview_path),
    embed=False,
    width=800
))

Previewing: uav0000086_00000_v
Video: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\uav0000086_00000_v.mp4
Frames: 464
FPS: 30.0
Resolution: 1344 x 756


## Section 6 — Check annotation frame alignment

The VisDrone MOT annotation files use rows with this structure:

`frame_index, target_id, bbox_left, bbox_top, bbox_width, bbox_height, score, object_category, truncation, occlusion`

The annotation frame number refers to the original sequence frame. The converter therefore saves `visdrone_frame_mapping.csv`, which maps the source image number to its new sequential MP4 frame number.

Keep the annotations separate from the clean input MP4. Do not draw ground-truth boxes onto the video you feed to YOLO, because those boxes would leak the answers into the input.

In [6]:
annotation_alignment_rows = []

for _, video_row in conversion_manifest.iterrows():
    sequence_name = str(video_row["video_id"])
    annotation_path = ANNOTATIONS_DIR / f"{sequence_name}.txt"

    sequence_mapping = frame_mapping[
        frame_mapping["video_id"] == sequence_name
    ].copy()

    if not annotation_path.exists() or annotation_path.stat().st_size == 0:
        annotation_alignment_rows.append({
            "video_id": sequence_name,
            "annotation_file_exists": annotation_path.exists(),
            "annotation_rows": 0,
            "annotation_frame_ids_found_in_images": 0,
            "status": "NO_ANNOTATION_FILE"
        })
        continue

    annotation = pd.read_csv(
        annotation_path,
        header=None,
        on_bad_lines="skip"
    )

    annotation_frame_ids = set(
        pd.to_numeric(
            annotation.iloc[:, 0],
            errors="coerce"
        ).dropna().astype(int)
    )

    source_frame_ids = set(
        pd.to_numeric(
            sequence_mapping["source_frame_number"],
            errors="coerce"
        ).dropna().astype(int)
    )

    matches = len(annotation_frame_ids.intersection(source_frame_ids))

    annotation_alignment_rows.append({
        "video_id": sequence_name,
        "annotation_file_exists": True,
        "annotation_rows": len(annotation),
        "unique_annotation_frames": len(annotation_frame_ids),
        "source_image_frames": len(source_frame_ids),
        "annotation_frame_ids_found_in_images": matches,
        "status": "CHECK" if matches == 0 and len(annotation) else "OK"
    })

annotation_alignment = pd.DataFrame(annotation_alignment_rows)
display(annotation_alignment)

ALIGNMENT_PATH = OUTPUT_DIR / "visdrone_annotation_alignment.csv"
annotation_alignment.to_csv(ALIGNMENT_PATH, index=False)

print("Annotation alignment report:", ALIGNMENT_PATH)

,video_id,annotation_file_exists,annotation_rows,unique_annotation_frames,source_image_frames,annotation_frame_ids_found_in_images,status
0,uav0000086_00000_v,True,23877,464,464,464,OK
1,uav0000117_02622_v,True,18463,349,349,349,OK
2,uav0000137_00458_v,True,24361,233,233,233,OK
3,uav0000182_00000_v,True,18441,363,363,363,OK
4,uav0000268_05773_v,True,13578,978,978,978,OK
5,uav0000305_00000_v,True,6391,184,184,184,OK
6,uav0000339_00001_v,True,13016,275,275,275,OK


Annotation alignment report: c:\Users\anyan\source\repos\AeroTrack\data\visdrone_reconstructed\visdrone_annotation_alignment.csv


## Section 7 — Files ready for AeroTrack

Your reconstructed videos are saved here:

`AeroTrack/data/visdrone_reconstructed/`

For example:

```text
AeroTrack/
├── data/
│   └── visdrone_reconstructed/
│       ├── uav0000086_00000_v.mp4
│       ├── uav0000117_02622_v.mp4
│       ├── uav0000137_00458_v.mp4
│       ├── uav0000182_00000_v.mp4
│       ├── uav0000268_05773_v.mp4
│       ├── uav0000305_00000_v.mp4
│       └── uav0000339_00001_v.mp4
│
└── outputs/
    ├── visdrone_video_manifest.csv
    ├── visdrone_frame_mapping.csv
    ├── visdrone_video_validation.csv
    └── visdrone_annotation_alignment.csv
```

### Use them in AeroTrack

1. Start with two or three MP4s, not all seven.
2. Run each MP4 through your existing Part 1 detection and BoT-SORT workflow.
3. Save a separate frame-level tracking CSV for every sequence.
4. Add the video and its tracking CSV to `VIDEO_SOURCES` in Part 11.
5. Keep the original annotation files and frame mapping for later evaluation.

### Important limitation for cross-video ReID

VisDrone MOT is useful for testing drone-view person detection and within-video tracking. Its target IDs are tracking identities inside their annotated sequence. Do **not** assume that target ID 12 in one sequence is the same real person as target ID 12 in another sequence.

The seven validation sequences are not guaranteed to contain the same people. Therefore, this dataset is useful for testing the multi-video pipeline and checking that unrelated people remain separate, but it does not by itself provide verified positive identity matches across different videos.

For a genuine returning-person cross-video test, use paired recordings known to show the same people from different views (for example, a suitable Okutama-Action paired scenario).

The VisDrone MOT benchmark is explicitly a multi-object tracking dataset built from drone video sequences, with annotations intended to describe trajectories across frames. See the [official VisDrone dataset documentation](https://github.com/VisDrone/VisDrone-Dataset).